In [26]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [27]:
print(os.listdir("/kaggle/input/zarrdata/zarrData"))

['val_ice_conc.zarr', 'test_ice_conc.zarr', 'pred_ice_conc.zarr', 'train_ice_conc.zarr']


In [28]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 78), started 0:15:57 ago. (Use '!kill 78' to kill it.)

<IPython.core.display.Javascript object>

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import os
import xarray as xr
import numpy as np
import glob
from datetime import datetime
import time
import torch.cuda.amp as amp

In [30]:
%pip install xarray[complete]

Note: you may need to restart the kernel to use updated packages.


In [31]:
import torch
import torch.nn as nn

class ConvLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size, bias):
        """
        input_size: channels of input tensor
        hidden_size: channels of hidden state
        kernel_size: size of the convolutional kernel
        bias: whether to use bias in the convolutional layer
        """
        super(ConvLSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.kernel_size = kernel_size
        self.bias = bias
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2

        self.conv = nn.Conv2d(in_channels=self.input_size + self.hidden_size,
                              out_channels=4 * self.hidden_size,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

        self.Wci = nn.Conv2d(in_channels=self.hidden_size, out_channels=self.hidden_size, kernel_size=1, bias=self.bias)
        self.Wcf = nn.Conv2d(in_channels=self.hidden_size, out_channels=self.hidden_size, kernel_size=1, bias=self.bias)
        self.Wco = nn.Conv2d(in_channels=self.hidden_size, out_channels=self.hidden_size, kernel_size=1, bias=self.bias)


    def forward(self, x, curr_state):
        h_cur, c_cur = curr_state
        combined = torch.cat([x, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv,
                                             split_size_or_sections=self.hidden_size,
                                             dim=1)
        i = torch.sigmoid(cc_i + self.Wci(c_cur))
        f = torch.sigmoid(cc_f + self.Wcf(c_cur))
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g

        o = torch.sigmoid(cc_o + self.Wco(c_next))

        h_next = o * torch.tanh(c_next)

        return h_next, c_next

class ConvLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size, num_layers, bias=True, return_sequences=False):
        super(ConvLSTM, self).__init__()
        self.input_size = input_size
        self.hidden_size = self._multilayer_support(hidden_size, num_layers)
        self.kernel_size = self._multilayer_support(kernel_size, num_layers)
        self.num_layers = num_layers
        self.bias = bias
        self.return_sequences = return_sequences
        cell_list = []

        for i in range(0, self.num_layers):
            curr_input_size = self.input_size if i == 0 else self.hidden_size[i-1]
            cell_list.append(ConvLSTMCell(input_size=curr_input_size,
                                          hidden_size=self.hidden_size[i],
                                          kernel_size=self.kernel_size[i],
                                          bias=self.bias))

        self.cell_list = nn.ModuleList(cell_list)

        self.output_projection = nn.Conv2d(in_channels=self.hidden_size[-1],
                                           out_channels=self.input_size,
                                           kernel_size=1,
                                           bias=self.bias)

        self.output_activation = nn.Sigmoid()

    def forward(self, x, hidden_state=None):
        b, seq_len, _, h, w = x.size()

        if hidden_state is None:
            hidden_state = self._init_hidden(b, (h, w), x.device)
        else:
            pass

        last_layer_output_sequence = []

        for t in range(seq_len):
            x_t = x[:,t,:,:,:]

            next_hidden_t = []

            #unrolling

            for idx in range(self.num_layers):
                cell = self.cell_list[idx]

                h_cur, c_cur = hidden_state[idx]
                h_next, c_next = cell(x=x_t, curr_state=[h_cur, c_cur])

                next_hidden_t.append((h_next, c_next))

                x_t = h_next

            hidden_state = next_hidden_t
            last_layer_output_sequence.append(x_t)

        last_layer_output_sequence = torch.stack(last_layer_output_sequence, dim = 0)
        #now seq_len is the first dimension
        last_layer_output_sequence = last_layer_output_sequence.permute(1, 0, 2, 3, 4)

        if self.return_sequences:
            projected_output = self.output_projection(last_layer_output_sequence.view(-1, self.hidden_size[-1], h, w))
            projected_output = projected_output.view(b, seq_len, self.input_size, h, w)
        else:
            projected_output = self.output_projection(last_layer_output_sequence[:,-1,:,:,:])


        final_prediction = self.output_activation(projected_output)

        return final_prediction


    def _init_hidden(self, batch_size, image_size, device):
        H, W = image_size
        init_states = []
        for i in range(self.num_layers):
            init_states.append((torch.zeros(batch_size, self.hidden_size[i], H, W, device=device),
                                torch.zeros(batch_size, self.hidden_size[i], H, W, device=device)))
        return init_states

    @staticmethod
    def _multilayer_support(x, num_layers):
        if (not isinstance(x, list)):
            x = [x] * num_layers
        return x


In [32]:
if torch.cuda.is_available():
  torch.cuda.empty_cache()
  print("CUDA cache cleared")

CUDA cache cleared


In [33]:
class IceConcDataset(Dataset):
  def __init__(self, zarr_path, sequence_length):
    self.zarr_path = zarr_path
    self.sequence_length = sequence_length

    print(f"Opening zarr store {zarr_path}")

    start_time = time.time()
    try:
      self.data_xr = xr.open_zarr(zarr_path)['ice_conc']
      print(f"Zarr store opened and stored in {time.time()-start_time:.2f} seconds")
      print(f"Lazy DataArray dask-backed shape: {self.data_xr.shape}")
    except Exception as e:
      print(f"ERROR: Could not open Zarr store at {zarr_path}: {e}")

    self.available_frames = len(self.data_xr['time'])
    print(f"Total number of frames available: {self.available_frames}")
    self.num_sequences = self.available_frames - self.sequence_length
    print(f"Total number of sequences available: {self.num_sequences}")

  def __len__(self):
    return self.num_sequences

  def __getitem__(self, idx):
    input_sequences = self.data_xr.isel(time=slice(idx, idx+self.sequence_length)).values
    output_frame = self.data_xr.isel(time=idx+self.sequence_length).values

    # T,H,W -> T,C,H,W
    input_tensor = torch.from_numpy(input_sequences).float().unsqueeze(1)

    # H,W -> C,H,W
    output_tensor = torch.from_numpy(output_frame).float().unsqueeze(0)

    return input_tensor, output_tensor



In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Core Hyperparameters
BATCH_SIZE = 2
IMAGE_SIZE = (432, 432)
LEARNING_RATE = 0.001
SEQUENCE_LENGTH = 3
EPOCHS = 20 # Increased epochs since we are sampling less data per epoch
NUM_LAYERS = 3
INPUT_CHANNELS = 1
HIDDEN_CHANNELS = [32, 64, 128] # Reduced for memory efficiency
KERNEL_SIZE = (3, 3)

LOG_INTERVAL = 5            # Log training loss every 5 batches
VAL_INTERVAL = 1            # Run validation every epoch
TRAIN_SAMPLE_PERCENT = 0.25 

Using device: cuda


In [35]:
current_time_str = datetime.now().strftime('%Y%m%d-%H%M%S')
LOG_DIR = f"runs/convlstm_ice_conc_forecasting_{current_time_str}"
os.makedirs(LOG_DIR, exist_ok=True)
writer = SummaryWriter(LOG_DIR)
print(f"TensorBoard logs will be saved to: {LOG_DIR}")

TensorBoard logs will be saved to: runs/convlstm_ice_conc_forecasting_20250711-100140


In [36]:
ZARR_DATA_PATH = '/kaggle/input/zarrdata/zarrData/'
SAVE_MODEL_DIR = '/kaggle/working/models/' # Directory to save models
os.makedirs(SAVE_MODEL_DIR, exist_ok=True) 

In [37]:
model = ConvLSTM(
    input_size=INPUT_CHANNELS,
    hidden_size=HIDDEN_CHANNELS,
    kernel_size=KERNEL_SIZE,
    num_layers=NUM_LAYERS
).to(device)

In [38]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = amp.GradScaler()

/tmp/ipykernel_36/2905668989.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


In [39]:
print(f"Model instantiated with {sum(p.numel() for p in model.parameters() if p.requires_grad)} trainable parameters.")

Model instantiated with 1210145 trainable parameters.


In [40]:
start_epoch = 0
best_val_loss = float('inf')
global_step = 0
checkpoint_to_load = os.path.join(SAVE_MODEL_DIR, "convlstm_best_model.pth")

if os.path.exists(checkpoint_to_load):
    try:
        checkpoint = torch.load(checkpoint_to_load, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_val_loss = checkpoint['best_val_loss']
        global_step = checkpoint['global_step']
        print(f"Resuming training from checkpoint: {checkpoint_to_load} (Epoch {start_epoch}, Best Val Loss: {best_val_loss:.6f})")
    except KeyError: 
        model.load_state_dict(torch.load(checkpoint_to_load, map_location=device))
        print(f"Model state_dict loaded from {checkpoint_to_load}. Resuming from scratch (epoch 0) with loaded weights.")
    except Exception as e:
        print(f"Error loading checkpoint {checkpoint_to_load}: {e}. Starting training from scratch.")
else:
    print("No checkpoint found, starting training from scratch.")


No checkpoint found, starting training from scratch.


In [41]:
def get_data_loaders(zarr_path, sequence_length, batch_size, train_sample_percent):
    train_zarr_path = os.path.join(zarr_path, "train_ice_conc.zarr")
    val_zarr_path = os.path.join(zarr_path, 'val_ice_conc.zarr')
    test_zarr_path = os.path.join(zarr_path, 'test_ice_conc.zarr')
    pred_zarr_path = os.path.join(zarr_path, 'pred_ice_conc.zarr')

    if not (os.path.exists(train_zarr_path) and os.path.exists(val_zarr_path) and os.path.exists(test_zarr_path)):
        print("\nERROR: Essential Zarr data stores not found.")
        print(f"Please ensure you have run `process_all_data_to_zarr()` first, and check the path: {zarr_path}")
        return None, None, None

    train_dataset = IceConcDataset(train_zarr_path, sequence_length)
    val_dataset = IceConcDataset(val_zarr_path, sequence_length)
    test_dataset = IceConcDataset(test_zarr_path, sequence_length)

    num_train_samples = len(train_dataset)
    num_sampled_train_indices = int(num_train_samples * train_sample_percent)

    all_train_indices = list(range(num_train_samples))
    random.shuffle(all_train_indices)
    sampled_train_indices = all_train_indices[:num_sampled_train_indices]
    
    train_subset = torch.utils.data.Subset(train_dataset, sampled_train_indices)

    train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=os.cpu_count()//2)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=os.cpu_count()//2) 
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=os.cpu_count()//2)

    print(f"Total training samples: {num_train_samples}")
    print(f"Number of sampled training samples (25%): {len(train_subset)}")
    print(f"Number of training batches (sampled): {len(train_dataloader)}")
    print(f"Number of validation batches: {len(val_dataloader)}")
    print(f"Number of test batches: {len(test_dataloader)}")

    return train_dataloader, val_dataloader, test_dataloader


In [42]:
def train_convlstm(model, optimizer, criterion, scaler, train_dataloader, val_dataloader, writer, num_epochs, log_interval, val_interval, save_dir, start_epoch_param, best_val_loss_param, global_step_param):
    global global_step, best_val_loss

    global_step = global_step_param
    best_val_loss = best_val_loss_param

    for epoch in range(start_epoch_param, num_epochs+1):
        model.train()
        running_loss = 0.0
        for batch_idx, (inputs, target) in enumerate(train_dataloader):
            inputs, target = inputs.to(device), target.to(device)
            optimizer.zero_grad()

            with amp.autocast():
                output = model(inputs)
                loss = criterion(output, target)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

            if (batch_idx+1) % log_interval == 0:
                avg_loss = running_loss/log_interval
                print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_dataloader)}], Loss: {loss.item():.4f}, Average Loss: {avg_loss:.6f}")
                writer.add_scalar('Loss/Train', avg_loss, global_step)
                running_loss = 0.
                
            global_step +=1

        if (epoch+1) % val_interval == 0:
            model.eval()
            val_loss = 0.
            with torch.no_grad():
                for inputs, target in val_dataloader:
                    inputs, target = inputs.to(device), target.to(device)

                    with amp.autocast():
                        output = model(inputs)
                        loss = criterion(output, target)
                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_dataloader)
            print(f"--- Epoch {epoch+1} Validation Loss: {avg_val_loss:.6f} ---")
            writer.add_scalar(f"Loss/Validation", avg_val_loss, global_step)

            checkpoint_data = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_val_loss': best_val_loss,
                'global_step': global_step,
            }

            torch.save(checkpoint_data, os.path.join(save_dir, f"convlstm_checkpoint_epoch{epoch+1}.pth"))
            print(f"Checkpoint for epoch {epoch+1} saved.")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_model_path = os.path.join(save_dir, f"convlstm_best_model.pth")
                torch.save(checkpoint_data, best_model_path)
                print(f"Best model saved to {best_model_path} (Validation Loss: {best_val_loss:.6f})")


            if len(val_dataloader)>0:

                sample_inputs, sample_targets = next(iter(val_dataloader)) # Get one batch
                sample_inputs_for_vis = sample_inputs[0:1].to(device)
                sample_targets_for_vis = sample_targets[0:1].to(device)

                with torch.no_grad():
                    sample_outputs = model(sample_inputs_for_vis)

                # Add images to TensorBoard
                # Ensure they are (C, H, W) for add_image. Your outputs are (1, H, W) -> (C=1, H, W) is fine.
                writer.add_image(f'Validation/Prediction_Epoch_{epoch}', sample_outputs[0], global_step)
                writer.add_image(f'Validation/GroundTruth_Epoch_{epoch}', sample_targets_for_vis[0], global_step)
                writer.add_image(f'Validation/Last_Input_Frame_Epoch_{epoch}', sample_inputs_for_vis[0, -1], global_step) # Last input frame
            else:
                print("Validation loader is empty, cannot add sample images to TensorBoard.")

    print("\nTraining Complete!")
    writer.close()
    print(f"TensorBoard logs are in: {LOG_DIR}")

            

In [43]:
import random

train_dataloader, val_dataloader, test_dataloader = get_data_loaders(
    ZARR_DATA_PATH, SEQUENCE_LENGTH, BATCH_SIZE, TRAIN_SAMPLE_PERCENT
)

Opening zarr store /kaggle/input/zarrdata/zarrData/train_ice_conc.zarr
Zarr store opened and stored in 0.03 seconds
Lazy DataArray dask-backed shape: (4633, 432, 432)
Total number of frames available: 4633
Total number of sequences available: 4630
Opening zarr store /kaggle/input/zarrdata/zarrData/val_ice_conc.zarr
Zarr store opened and stored in 0.02 seconds
Lazy DataArray dask-backed shape: (731, 432, 432)
Total number of frames available: 731
Total number of sequences available: 728
Opening zarr store /kaggle/input/zarrdata/zarrData/test_ice_conc.zarr
Zarr store opened and stored in 0.02 seconds
Lazy DataArray dask-backed shape: (730, 432, 432)
Total number of frames available: 730
Total number of sequences available: 727
Total training samples: 4630
Number of sampled training samples (25%): 1157
Number of training batches (sampled): 579
Number of validation batches: 364
Number of test batches: 364


In [44]:
if train_dataloader is not None: # Only proceed if data loaders were successfully created
    # Start training
    train_convlstm(model, optimizer, criterion, scaler,
                   train_dataloader, val_dataloader, writer,
                   EPOCHS, LOG_INTERVAL, VAL_INTERVAL, SAVE_MODEL_DIR,
                   start_epoch, best_val_loss, global_step)

/tmp/ipykernel_36/3141991092.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


Epoch [1/20], Batch [5/579], Loss: 0.2043, Average Loss: 0.223891
Epoch [1/20], Batch [10/579], Loss: 0.0386, Average Loss: 0.109647
Epoch [1/20], Batch [15/579], Loss: 0.0149, Average Loss: 0.022791
Epoch [1/20], Batch [20/579], Loss: 0.0117, Average Loss: 0.015225
Epoch [1/20], Batch [25/579], Loss: 0.0141, Average Loss: 0.011860
Epoch [1/20], Batch [30/579], Loss: 0.0089, Average Loss: 0.011376
Epoch [1/20], Batch [35/579], Loss: 0.0062, Average Loss: 0.011449
Epoch [1/20], Batch [40/579], Loss: 0.0047, Average Loss: 0.006827
Epoch [1/20], Batch [45/579], Loss: 0.0052, Average Loss: 0.006949
Epoch [1/20], Batch [50/579], Loss: 0.0048, Average Loss: 0.005903
Epoch [1/20], Batch [55/579], Loss: 0.0059, Average Loss: 0.006689
Epoch [1/20], Batch [60/579], Loss: 0.0085, Average Loss: 0.006161
Epoch [1/20], Batch [65/579], Loss: 0.0040, Average Loss: 0.006077
Epoch [1/20], Batch [70/579], Loss: 0.0053, Average Loss: 0.004895
Epoch [1/20], Batch [75/579], Loss: 0.0026, Average Loss: 0.004

/tmp/ipykernel_36/3141991092.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


--- Epoch 1 Validation Loss: 0.001039 ---
Checkpoint for epoch 1 saved.
Best model saved to /kaggle/working/models/convlstm_best_model.pth (Validation Loss: 0.001039)
Epoch [2/20], Batch [5/579], Loss: 0.0010, Average Loss: 0.000877
Epoch [2/20], Batch [10/579], Loss: 0.0009, Average Loss: 0.000890
Epoch [2/20], Batch [15/579], Loss: 0.0010, Average Loss: 0.000868
Epoch [2/20], Batch [20/579], Loss: 0.0006, Average Loss: 0.000845
Epoch [2/20], Batch [25/579], Loss: 0.0008, Average Loss: 0.000881
Epoch [2/20], Batch [30/579], Loss: 0.0010, Average Loss: 0.000839
Epoch [2/20], Batch [35/579], Loss: 0.0007, Average Loss: 0.000815
Epoch [2/20], Batch [40/579], Loss: 0.0005, Average Loss: 0.000803
Epoch [2/20], Batch [45/579], Loss: 0.0008, Average Loss: 0.000870
Epoch [2/20], Batch [50/579], Loss: 0.0007, Average Loss: 0.000796
Epoch [2/20], Batch [55/579], Loss: 0.0007, Average Loss: 0.000778
Epoch [2/20], Batch [60/579], Loss: 0.0010, Average Loss: 0.000809
Epoch [2/20], Batch [65/579], 